# Encounter Anomaly Detection

Pulls the anomaly marts built in dbt (`mart_encounter_cost_anomalies`, `mart_monthly_encounter_anomalies`), visualizes the flagged encounters/months, and exports all mart CSVs to `exports/` for Power BI.

Run standalone in Jupyter, or headlessly via `python scripts/run_pipeline.py` (which runs this with papermill after refreshing the data).

In [ ]:
# papermill parameters — override with -p when running headlessly
EXPORT_DIR = "exports"

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import psycopg2

EXPORT_DIR = Path(EXPORT_DIR)
EXPORT_DIR.mkdir(exist_ok=True)


def get_conn():
    return psycopg2.connect(
        host=os.getenv("PGHOST", "localhost"),
        port=int(os.getenv("PGPORT", 5432)),
        dbname=os.getenv("PGDATABASE", "healthcare"),
        user=os.getenv("PGUSER", "postgres"),
        password=os.getenv("PGPASSWORD", "postgres"),
    )


MARTS = [
    "mart_patient_demographics",
    "mart_encounter_trends",
    "mart_condition_prevalence",
    "mart_medication_utilization",
    "mart_encounter_cost_anomalies",
    "mart_monthly_encounter_anomalies",
]

conn = get_conn()
marts = {name: pd.read_sql(f"SELECT * FROM marts.{name}", conn) for name in MARTS}
conn.close()

cost_anomalies = marts["mart_encounter_cost_anomalies"]
volume_anomalies = marts["mart_monthly_encounter_anomalies"]

for name, df in marts.items():
    print(f"{name:<40} {len(df):>8,} rows")

## Cost anomalies

Encounters flagged in `mart_encounter_cost_anomalies` (`is_cost_anomaly = true`) are ≥ 3 standard deviations from the average cost of their peer group (same encounter class + type).

In [ ]:
flagged = cost_anomalies[cost_anomalies["is_cost_anomaly"]]
print(f"{len(flagged):,} of {len(cost_anomalies):,} encounters flagged "
      f"({len(flagged) / len(cost_anomalies):.2%})")

flagged.sort_values("cost_z_score", ascending=False).head(20)[
    ["encounter_id", "encounter_type", "state", "total_claim_cost",
     "peer_avg_cost", "cost_z_score"]
]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = cost_anomalies["is_cost_anomaly"].map({True: "crimson", False: "steelblue"})
ax.scatter(cost_anomalies["total_claim_cost"], cost_anomalies["cost_z_score"],
           c=colors, alpha=0.4, s=12)
ax.axhline(3, color="black", linestyle="--", linewidth=1, label="z = 3 threshold")
ax.set_xlabel("Total claim cost ($)")
ax.set_ylabel("Cost z-score vs. peer group")
ax.set_title("Encounter cost anomalies")
ax.legend()
plt.show()

In [ ]:
flagged.groupby("encounter_type").size().sort_values(ascending=False).head(10).plot(
    kind="barh", figsize=(8, 5), title="Flagged cost anomalies by encounter type"
)
plt.xlabel("Flagged encounters")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Volume anomalies

Months flagged in `mart_monthly_encounter_anomalies` (`is_volume_anomaly = true`) deviate ≥ 2 standard deviations from the trailing 6-month rolling average for that encounter class.

In [ ]:
top_classes = (
    volume_anomalies.groupby("encounter_class")["encounter_count"]
    .sum().sort_values(ascending=False).head(5).index
)

fig, axes = plt.subplots(len(top_classes), 1, figsize=(9, 3 * len(top_classes)), sharex=False)
if len(top_classes) == 1:
    axes = [axes]

for ax, cls in zip(axes, top_classes):
    sub = volume_anomalies[volume_anomalies["encounter_class"] == cls].sort_values("encounter_month")
    ax.plot(sub["encounter_month"], sub["encounter_count"], color="steelblue", label="encounter_count")
    anomalous = sub[sub["is_volume_anomaly"]]
    ax.scatter(anomalous["encounter_month"], anomalous["encounter_count"],
               color="crimson", zorder=5, label="flagged")
    ax.set_title(f"{cls} — monthly encounter volume")
    ax.legend()

plt.tight_layout()
plt.show()

## Export for Power BI

In [ ]:
for name, df in marts.items():
    out = EXPORT_DIR / f"{name}.csv"
    df.to_csv(out, index=False)
    print(f"{name:<40} {len(df):>8,} rows  ->  {out}")

print("\nDone. Get Data -> Text/CSV in Power BI, pointed at the exports/ folder.")